# Step 1b: pyannote finds the speakers (GPU)

Kernel: **PyAnnote Diarization** (`~/envs/pyannote-diar`, see `../../environments/README.md`). Run it after step 1a, with the same `MEETING` and `AUDIO` in the first line. The cell reads `HF_TOKEN` from `pipeline/.env`. The account of the token must have accepted the conditions of `pyannote/speaker-diarization-community-1`.

The cell finds the speaker turns, gives every word the speaker active at its midpoint and writes the three files `protokoll.py` reads into `outputs/<MEETING>/`: `transcript_speakers.txt`, `transcript_canary_ts.json` and `diarization_hyp.rttm`. It also writes `goldset_segments_canary.json`, which step 2 does not use. Steps 1a and 1b together run the same functions in the same order as `thesis_pipeline.canary_diarize.process_meeting()`, split over two environments.

In [ ]:
MEETING, AUDIO = "april27", "data/Team-Meeting_2026-04-27.flac"

import os, sys, json
from pathlib import Path
from dataclasses import asdict
ROOT = Path.cwd().parent if Path.cwd().name == "cluster" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, os.getcwd())
from thesis_pipeline.env import load_env
from thesis_pipeline.canary_diarize import (Word, diarize, assign_speakers,
    speaker_attributed_transcript, write_rttm, build_speaker_pure_segments)

load_env()
out = Path("outputs") / MEETING
wav = str(out / (Path(AUDIO).stem + "_16k_mono.wav"))
words = [Word(**w) for w in json.loads((out / "_words_step1a.json").read_text(encoding="utf-8"))]
turns = diarize(wav, hf_token=os.environ["HF_TOKEN"])
assign_speakers(words, turns)
(out / "transcript_canary_ts.json").write_text(json.dumps([asdict(w) for w in words], ensure_ascii=False, indent=2), encoding="utf-8")
(out / "transcript_speakers.txt").write_text(speaker_attributed_transcript(words), encoding="utf-8")
write_rttm(turns, MEETING, out / "diarization_hyp.rttm")
segs = build_speaker_pure_segments(words, MEETING)
(out / "goldset_segments_canary.json").write_text(json.dumps([asdict(s) for s in segs], ensure_ascii=False, indent=2), encoding="utf-8")
print({"words": len(words), "turns": len(turns), "speakers": len({t.speaker for t in turns}), "segments": len(segs)})